# sg-sem: build the data and train the model

The Skill Graph semantic checker (see `SKILL_GRAPH_SEMANTIC_MODEL_SPEC.md` in the repo). This notebook runs on a **Colab GPU** through the VS Code Colab extension:

1. clones the repo and runs our own graph library (Node) to build seed graphs and text jobs,
2. runs an open model **on this GPU** (no API) to write outlines, checker answers, plain-word phrases and gates,
3. assembles the labelled examples, split by graph,
4. fine-tunes the model and reports dev and test metrics against the 0.9 precision bar.

**Run it with `QUICK_LOOK = True` first** (about 10 minutes): it writes a few answers per round so their quality can be checked. Then set it to `False` and run all again; finished answers are kept, so nothing is redone.

Save the notebook when it finishes: its outputs are what gets reviewed.

In [7]:
REPO = "https://github.com/Saad-web-spec/DevScratchPad.git"
BRANCH = "sg-sem"  # the branch with the sg-sem code
GENERATOR = "Qwen/Qwen2.5-7B-Instruct"   # writes the text; runs here on the GPU
BASE = "nreimers/MiniLM-L12-H384-uncased"  # the encoder sg-sem is fine-tuned from
QUICK_LOOK = True                          # True: a few answers per round, to check quality first
TRAIN = not QUICK_LOOK                     # training waits for the full data
RUN = "minilm-1"

## 1. Storage and setup

In [8]:
import os, shutil, subprocess
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/sg-sem"
except Exception as error:  # the VS Code extension may not support Drive mounting
    print("Google Drive not mounted:", error)
    print("Data stays on this runtime only: if it disconnects, the run starts again.")
    ROOT = "/content/sg-sem"
DATA = f"{ROOT}/data-quick" if QUICK_LOOK else f"{ROOT}/data"
RUNS = f"{ROOT}/runs"
os.makedirs(DATA, exist_ok=True); os.makedirs(RUNS, exist_ok=True)
print("data:", DATA)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
data: /content/drive/MyDrive/sg-sem/data-quick


In [9]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
CODE = "/content/DevScratchPad"
if not os.path.exists(CODE):
    !git clone --depth 1 -b {BRANCH} {REPO} {CODE}
else:
    !git -C {CODE} pull --ff-only
!git -C {CODE} log --oneline -1
node = subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip() if shutil.which("node") else ""
if not node.startswith(("v20", "v22", "v24")):
    !curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null 2>&1 && apt-get install -y nodejs > /dev/null 2>&1
!node --version
%cd {CODE}
!npm ci --ignore-scripts --no-audit --no-fund --loglevel=error
!pip install -q -r ml/sg-sem/requirements.txt
PIPE = "npx vite-node scripts/semantic/pipeline.ts"
LIMIT = "--limit 24" if QUICK_LOOK else ""

name, memory.total [MiB]
NVIDIA L4, 23034 MiB
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 9 (delta 6), reused 9 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 1.26 KiB | 431.00 KiB/s, done.
From https://github.com/Saad-web-spec/DevScratchPad
   dcb753f..d08111a  sg-sem     -> origin/sg-sem
Updating dcb753f..d08111a
Fast-forward
 ml/sg-sem/sgsem/generate.py  |  2 ++
 scripts/semantic/pipeline.ts | 24 ++++++++++++++----------
 2 files changed, 16 insertions(+), 10 deletions(-)
d08111a (HEAD -> sg-sem, origin/sg-sem) fix(sg-sem): pipeline runs on Linux (Colab); a missing step's file fails loudly
v20.19.0
/content/DevScratchPad
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
added 557 packages in 13s
⠋

## 2. Round 1: workflow outlines → seed graphs

In [10]:
!{PIPE} round1 --out {DATA}
!cd ml/sg-sem && python -m sgsem.generate --requests {DATA}/round1.requests.jsonl --outputs {DATA}/round1.outputs.jsonl --model {GENERATOR} {LIMIT}

⠙round1: 200 outline requests → /content/drive/MyDrive/sg-sem/data-quick/round1.requests.jsonl
⠙0 already answered, 24 to go
tokenizer_config.json: 7.30kB [00:00, 27.9MB/s]
vocab.json: 2.78MB [00:00, 122MB/s]
merges.txt: 1.67MB [00:00, 140MB/s]
tokenizer.json: 7.03MB [00:00, 190MB/s]
config.json: 100% 663/663 [00:00<00:00, 6.15MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1792676981.351772    4581 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792676981.424053    4581 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792676

## 3. Round 2: checker answers, plain-word inputs, gates

In [11]:
!{PIPE} round2 --out {DATA}
!cd ml/sg-sem && python -m sgsem.generate --requests {DATA}/round2.requests.jsonl --outputs {DATA}/round2.outputs.jsonl --model {GENERATOR} {LIMIT}

⠙round2: 16 seeds (2 drafted; rejected {"json":0,"draft":22,"errors":0,"duplicate":0}), 175 requests {"h1":144,"h4":18,"h2":13}
⠙0 already answered, 24 to go
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1792677181.931414    5733 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792677182.000555    5733 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792677184.599735    5733 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computati

## 4. Assemble the labelled examples

In [12]:
!{PIPE} assemble --out {DATA}
print(open(f"{DATA}/stats.json").read())

⠙assemble: 222 examples from 16 seeds; dropped {"h1-unreadable-resolved_problem_pass":1,"h1-unreadable-no_evidence_pass":3,"h1-unreadable-honest_pass":2,"h1-unreadable-buried_problem_pass":2}
{"h1/test":{"label0":3,"label1":2},"h1/train":{"label0":4,"label1":1},"h4/test":{"label0":5,"label1":2,"label2":3},"h2/test":{"label1":2,"label0":12},"h2/train":{"label1":2,"label0":74},"h3/test":{"label0":5,"label1":5},"h4/train":{"label0":11},"h3/train":{"label0":40,"label1":37},"h2/dev":{"label0":7},"h4/dev":{"label0":1},"h3/dev":{"label0":3,"label1":3}}
⠙{
  "input_version": 1,
  "seeds": 16,
  "examples": 222,
  "by_head_split": {
    "h1/test": {
      "label0": 3,
      "label1": 2
    },
    "h1/train": {
      "label0": 4,
      "label1": 1
    },
    "h4/test": {
      "label0": 5,
      "label1": 2,
      "label2": 3
    },
    "h2/test": {
      "label1": 2,
      "label0": 12
    },
    "h2/train": {
      "label1": 2,
      "label0": 74
    },
    "h3/test": {
      "label0": 5,
    

In [13]:
# For review: raw answers of the text model, two per kind, and the assembled samples.
import json, collections
requests = {json.loads(l)["id"]: json.loads(l) for l in open(f"{DATA}/round2.requests.jsonl")}
shown = collections.Counter()
for line in open(f"{DATA}/round2.outputs.jsonl"):
    row = json.loads(line); kind = requests[row["id"]]["kind"]
    if shown[kind] < 2:
        shown[kind] += 1
        print(f"--- {kind} {requests[row['id']]['meta']}")
        print(row["text"][:1200])
        print()
print(open(f"{DATA}/samples.md").read()[:8000])

--- h1 {'style': 'resolved_problem_pass', 'seed': 'research-and-fact-check', 'node': 'fact_check'}
{"sentences":[{"text":"After reviewing the claims and their sources, all the remaining statements were supported by the documents provided, with no contradictory information found from other reliable sources.","open_problem":false}]}

--- h1 {'style': 'resolved_problem_pass', 'seed': 'research-and-fact-check', 'node': 'fact_check'}
{"sentences": [{"text": "After reviewing the claims, I found all kept claims were backed by their sources, meeting the requirement that each kept claim must have a supporting source.", "open_problem": false}, {"text": "Additionally, no kept claim was contradicted by a more reliable or a more recent source, fulfilling the criteria for accuracy and reliability.", "open_problem": false}, {"text": "Each source was carefully checked against the claims, ensuring that the information provided was accurate and up-to-date.", "open_problem": false}, {"text": "For instanc

## 5. Train (runs when `QUICK_LOOK = False`)

In [14]:
if TRAIN:
    !cd ml/sg-sem && python -m sgsem.train --data {DATA}/examples.jsonl --out {RUNS}/{RUN} --base {BASE}
else:
    print("QUICK_LOOK is on: training waits for the full data.")

QUICK_LOOK is on: training waits for the full data.


In [15]:
if TRAIN:
    report = json.load(open(f"{RUNS}/{RUN}/metrics.json"))
    print(json.dumps({k: report[k] for k in ["test", "rules_baseline_test", "meets_bar_on_test", "minutes"]}, indent=1))
    print(open(f"{RUNS}/{RUN}/errors.md").read()[:6000])